# San Fernando ITT - Geovisor Interactivo

**Flujo de trabajo:**
1. Configuracion del entorno (Colab)
2. Carga y transformacion a WGS84
3. Validacion y exportacion a GeoJSON
4. Geovisor interactivo (poligono editable)
5. Definir area de estudio
6. (OPCIONAL) Subir poligono editado
7. Extraccion de POIs de OpenStreetMap
8. Tabla de POIs
9. Mapa final con POIs
10. Exportacion de resultados

## 1. Configuracion del entorno

In [ ]:
import sys
import subprocess
from pathlib import Path

EN_COLAB = 'google.colab' in sys.modules

if EN_COLAB:
    try:
        import mapclassify
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mapclassify'])
    import shutil
    if Path('Corredor_San_fernando').exists():
        shutil.rmtree('Corredor_San_fernando')
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/j0rg3c45/Corredor_San_fernando.git'])
    ruta_base = Path('Corredor_San_fernando')
else:
    ruta_base = Path('..')

print(f"Entorno: {'Google Colab' if EN_COLAB else 'Local'}")
print(f"Ruta base: {ruta_base.resolve()}")

## 2. Importaciones y rutas

In [ ]:
import geopandas as gpd
import pandas as pd
import folium
from folium.plugins import Draw
import json
import requests

# Rutas del proyecto
ruta_geojson_fuente = ruta_base / 'Data' / 'Geojson_Corredor_San_fernando' / 'geojson_Corredor_San_fernando.geojson'
ruta_shapefile = ruta_base / 'Data' / 'shape_geo' / 'Corredor_San_fernando.shp'
ruta_geojson_wgs84 = ruta_base / 'Data' / 'Corredor_San_fernando_wgs84.geojson'
ruta_outputs = ruta_base / 'Outputs'
ruta_outputs.mkdir(parents=True, exist_ok=True)

# Priorizar GeoJSON actualizado sobre shapefile
if ruta_geojson_fuente.exists():
    ruta_entrada = ruta_geojson_fuente
elif ruta_shapefile.exists():
    ruta_entrada = ruta_shapefile
else:
    raise FileNotFoundError('No se encontro archivo de entrada en Data/')

print(f"Archivo de entrada: {ruta_entrada}")

## 3. Carga, transformacion y validacion

In [ ]:
# Cargar datos
gdf_corredor = gpd.read_file(ruta_entrada)
print(f"CRS original: {gdf_corredor.crs}")
print(f"Registros: {len(gdf_corredor)}")

# Asignar CRS si no tiene
if gdf_corredor.crs is None:
    gdf_corredor = gdf_corredor.set_crs(epsg=3115)

# Reproyectar a WGS84
if gdf_corredor.crs.to_epsg() != 4326:
    gdf_corredor = gdf_corredor.to_crs(epsg=4326)

# Validar geometrias
invalidas = gdf_corredor[~gdf_corredor.geometry.is_valid]
if len(invalidas) > 0:
    gdf_corredor['geometry'] = gdf_corredor.geometry.buffer(0)
    print(f"Corregidas {len(invalidas)} geometrias invalidas.")

# Exportar GeoJSON en WGS84
gdf_corredor.to_file(ruta_geojson_wgs84, driver='GeoJSON')

print(f"CRS final: {gdf_corredor.crs}")
print(f"Bounds: {gdf_corredor.total_bounds}")
gdf_corredor.head()

## 4. Geovisor interactivo (poligono editable)

El poligono se carga dentro del FeatureGroup editable.
- Haz clic en el icono de edicion (lapiz) para mover vertices
- Haz clic en Save cuando termines
- Haz clic en Export para descargar el poligono editado
- Sube ese archivo en la celda 6 (opcional)

In [ ]:
# Leer GeoJSON
with open(ruta_geojson_wgs84) as f:
    geojson_data = json.load(f)

# Centro del mapa
centroide = gdf_corredor.union_all().centroid
centro = [centroide.y, centroide.x]

# Crear mapa
mapa = folium.Map(location=centro, zoom_start=15, tiles='OpenStreetMap')
folium.TileLayer('CartoDB positron', name='CartoDB Claro').add_to(mapa)
folium.TileLayer('CartoDB dark_matter', name='CartoDB Oscuro').add_to(mapa)

# FeatureGroup editable
fg_editable = folium.FeatureGroup(name='Poligono editable')
fg_editable.add_to(mapa)

# Draw plugin
Draw(
    export=True,
    draw_options={'polyline': False, 'rectangle': True, 'polygon': True,
                  'circle': False, 'marker': False, 'circlemarker': False},
    edit_options={'edit': True, 'remove': True}
).add_to(mapa)

# Inyectar JS para cargar poligono en el FeatureGroup editable
geojson_str = json.dumps(geojson_data)
js_code = f"""
<script>
setTimeout(function() {{
    var map = null;
    document.querySelectorAll('.folium-map').forEach(function(el) {{
        if (window[el.id]) map = window[el.id];
    }});
    if (!map) return;
    var drawnItems = null;
    for (var key in window) {{
        if (key.startsWith('drawnItems_')) {{ drawnItems = window[key]; break; }}
    }}
    if (drawnItems) {{
        L.geoJSON({geojson_str}).eachLayer(function(layer) {{
            drawnItems.addLayer(layer);
        }});
    }}
}}, 1000);
</script>
"""
mapa.get_root().html.add_child(folium.Element(js_code))
folium.LayerControl(collapsed=False).add_to(mapa)

# Guardar HTML
ruta_mapa = ruta_outputs / 'geovisor_editable.html'
mapa.save(str(ruta_mapa))
print(f"Mapa guardado: {ruta_mapa}")
mapa

## 5. Definir area de estudio

Se usa el poligono original. Si quieres usar un poligono editado,
ejecuta la celda 6 (opcional) ANTES de esta.

In [ ]:
# Usar poligono original como area de estudio
# Si ejecutaste la celda 6 y subiste un poligono, esta variable ya estara definida
if 'gdf_area_estudio' not in dir():
    gdf_area_estudio = gdf_corredor.copy()

print(f"Area de estudio: {len(gdf_area_estudio)} poligono(s)")
print(f"Bounds: {gdf_area_estudio.total_bounds}")

## 6. (OPCIONAL) Subir poligono editado

Ejecuta esta celda SOLO si editaste el poligono en el mapa y quieres usarlo.
Si no, **saltala** y ve directo a la seccion 7.

In [ ]:
from google.colab import files

print("Sube el GeoJSON editado (data.geojson):")
uploaded = files.upload()

if uploaded:
    nombre_archivo = list(uploaded.keys())[0]
    contenido = uploaded[nombre_archivo].decode('utf-8')
    geojson_editado = json.loads(contenido)
    gdf_area_estudio = gpd.GeoDataFrame.from_features(
        geojson_editado['features'], crs='EPSG:4326'
    )
    print(f"Poligono editado cargado: {len(gdf_area_estudio)} geometria(s)")
    print(f"Bounds: {gdf_area_estudio.total_bounds}")

## 7. Extraccion de POIs de OpenStreetMap

Se usa Overpass API (POST) para extraer puntos de interes dentro del area.

In [ ]:
def consultar_overpass(bbox, categorias):
    """Consulta Overpass API via POST. bbox: (sur, oeste, norte, este)"""
    url = 'https://overpass-api.de/api/interpreter'
    resultados = []

    for categoria, tags in categorias.items():
        filtros_node = ''.join(
            f'node["{k}"="{v}"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});'
            for k, v in tags
        )
        filtros_way = ''.join(
            f'way["{k}"="{v}"]({bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]});'
            for k, v in tags
        )
        query = f'[out:json][timeout:60];({filtros_node}{filtros_way});out center;'
        resp = requests.post(url, data={'data': query})
        if resp.status_code == 200:
            datos = resp.json()
            for elem in datos.get('elements', []):
                lat = elem.get('lat') or elem.get('center', {}).get('lat')
                lon = elem.get('lon') or elem.get('center', {}).get('lon')
                if lat and lon:
                    nombre = elem.get('tags', {}).get('name', 'Sin nombre')
                    subtipo = elem.get('tags', {}).get(tags[0][0], '')
                    resultados.append({
                        'nombre': nombre,
                        'categoria': categoria,
                        'subtipo': subtipo,
                        'latitud': lat,
                        'longitud': lon,
                        'osm_id': elem.get('id')
                    })
        else:
            print(f"Error '{categoria}': HTTP {resp.status_code}")

    return pd.DataFrame(resultados)


# Categorias de POIs
categorias_poi = {
    'Salud': [('amenity', 'hospital'), ('amenity', 'clinic'),
              ('amenity', 'pharmacy'), ('amenity', 'doctors')],
    'Educacion': [('amenity', 'school'), ('amenity', 'university'),
                  ('amenity', 'kindergarten'), ('amenity', 'college')],
    'Comercio': [('shop', 'supermarket'), ('shop', 'convenience'),
                 ('shop', 'mall'), ('amenity', 'marketplace')],
    'Transporte': [('amenity', 'bus_station'), ('highway', 'bus_stop'),
                   ('amenity', 'fuel'), ('amenity', 'parking')],
    'Recreacion': [('leisure', 'park'), ('leisure', 'sports_centre'),
                   ('leisure', 'playground'), ('amenity', 'community_centre')],
    'Servicios': [('amenity', 'bank'), ('amenity', 'police'),
                  ('amenity', 'fire_station'), ('amenity', 'post_office')]
}

# Bounding box del area de estudio
bounds = gdf_area_estudio.total_bounds
bbox_overpass = (float(bounds[1]), float(bounds[0]), float(bounds[3]), float(bounds[2]))

print(f"Bbox: {bbox_overpass}")
print("Consultando Overpass API...")
df_pois = consultar_overpass(bbox_overpass, categorias_poi)
print(f"Total POIs encontrados: {len(df_pois)}")

## 8. Filtrar POIs dentro del poligono y mostrar tabla

In [ ]:
if len(df_pois) > 0:
    # Crear GeoDataFrame
    gdf_pois = gpd.GeoDataFrame(
        df_pois,
        geometry=gpd.points_from_xy(df_pois.longitud, df_pois.latitud),
        crs='EPSG:4326'
    )
    # Filtrar solo dentro del poligono
    gdf_pois_dentro = gpd.sjoin(
        gdf_pois, gdf_area_estudio[['geometry']], predicate='within', how='inner'
    ).drop(columns=['index_right'], errors='ignore')

    print(f"POIs dentro del poligono: {len(gdf_pois_dentro)}")
    print(f"\nPor categoria:")
    print(gdf_pois_dentro['categoria'].value_counts().to_string())

    # Tabla
    tabla = gdf_pois_dentro[['nombre', 'categoria', 'subtipo', 'latitud', 'longitud']]
    tabla = tabla.sort_values(['categoria', 'nombre']).reset_index(drop=True)
    tabla.index = tabla.index + 1
    tabla.index.name = 'N'
    print(f"\n--- Tabla de POIs ---")
    display(tabla)
else:
    print("No se encontraron POIs.")
    gdf_pois_dentro = gpd.GeoDataFrame()

## 9. Mapa final con POIs clasificados

In [ ]:
colores = {'Salud': 'red', 'Educacion': 'blue', 'Comercio': 'green',
           'Transporte': 'orange', 'Recreacion': 'purple', 'Servicios': 'darkblue'}

mapa_final = folium.Map(location=centro, zoom_start=15, tiles='CartoDB positron')

# Poligono del area
folium.GeoJson(
    gdf_area_estudio.to_json(),
    name='Area de estudio',
    style_function=lambda x: {'fillColor': '#3388ff', 'color': '#2255cc',
                               'weight': 2, 'fillOpacity': 0.1}
).add_to(mapa_final)

# POIs por categoria
if len(gdf_pois_dentro) > 0:
    for cat, color in colores.items():
        pois_cat = gdf_pois_dentro[gdf_pois_dentro['categoria'] == cat]
        if len(pois_cat) == 0:
            continue
        fg = folium.FeatureGroup(name=f"{cat} ({len(pois_cat)})")
        for _, poi in pois_cat.iterrows():
            folium.CircleMarker(
                location=[poi.latitud, poi.longitud],
                radius=6, color=color, fill=True,
                fill_color=color, fill_opacity=0.7,
                popup=f"<b>{poi.nombre}</b><br>{poi.categoria} - {poi.subtipo}",
                tooltip=poi.nombre
            ).add_to(fg)
        fg.add_to(mapa_final)

folium.LayerControl(collapsed=False).add_to(mapa_final)
mapa_final

## 10. Exportacion de resultados

In [ ]:
from google.colab import files as colab_files

if len(gdf_pois_dentro) > 0:
    # GeoJSON de POIs
    ruta_pois_gj = ruta_outputs / 'pois_area_estudio.geojson'
    gdf_pois_dentro.to_file(ruta_pois_gj, driver='GeoJSON')
    print(f"POIs GeoJSON: {ruta_pois_gj}")

    # CSV de POIs
    ruta_pois_csv = ruta_outputs / 'pois_area_estudio.csv'
    gdf_pois_dentro.drop(columns='geometry').to_csv(ruta_pois_csv, index=False)
    print(f"POIs CSV: {ruta_pois_csv}")

# Mapa HTML
ruta_html = ruta_outputs / 'geovisor_pois_san_fernando.html'
mapa_final.save(str(ruta_html))
print(f"Mapa HTML: {ruta_html}")

# Descargar en Colab
if EN_COLAB:
    if len(gdf_pois_dentro) > 0:
        colab_files.download(str(ruta_pois_csv))
    colab_files.download(str(ruta_html))